In [24]:
!pip install pandas numpy scikit-learn matplotlib seaborn mlflow scikit-learn

print("Required libraries installed successfully.")

Required libraries installed successfully.


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [27]:
import pandas as pd
df = pd.read_csv('/content/sample_data/NY_Trip_Raw_Data.csv')
print("Data loaded successfully. First 5 rows:")
print(df.head())

Data loaded successfully. First 5 rows:
          id  vendor_id  pickup_datetime dropoff_datetime  passenger_count  \
0  id2875421          2  3/14/2016 17:24  3/14/2016 17:32                1   
1  id2377394          1   6/12/2016 0:43   6/12/2016 0:54                1   
2  id3858529          2  1/19/2016 11:35  1/19/2016 12:10                1   
3  id3504673          2   4/6/2016 19:32   4/6/2016 19:39                1   
4  id2181028          2  3/26/2016 13:30  3/26/2016 13:38                1   

    weather  pickup_longitude  pickup_latitude  dropoff_longitude  \
0     sunny        -73.982155        40.767937         -73.964630   
1     snowy        -73.980415        40.738564         -73.999481   
2  freezing        -73.979027        40.763939         -74.005333   
3     rainy        -74.010040        40.719971         -74.012268   
4     windy        -73.973053        40.793209         -73.972923   

   dropoff_latitude store_and_fwd_flag  trip_duration  
0         40.765602 

In [28]:
print("DataFrame Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDescriptive Statistics:")
print(df.describe())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 12 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1048575 non-null  object 
 1   vendor_id           1048575 non-null  int64  
 2   pickup_datetime     1048575 non-null  object 
 3   dropoff_datetime    1048575 non-null  object 
 4   passenger_count     1048575 non-null  int64  
 5   weather             1048575 non-null  object 
 6   pickup_longitude    1048575 non-null  float64
 7   pickup_latitude     1048575 non-null  float64
 8   dropoff_longitude   1048575 non-null  float64
 9   dropoff_latitude    1048575 non-null  float64
 10  store_and_fwd_flag  1048575 non-null  object 
 11  trip_duration       1048575 non-null  int64  
dtypes: float64(4), int64(3), object(5)
memory usage: 96.0+ MB

Missing Values:
id                    0
vendor_id             0
pickup_datetime       0
dropo



---



In [29]:

# --- 1. Schema Validation ---
print("\n--- Schema Validation ---")

# Convert datetime columns
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'], errors='coerce')

# Check for unparseable dates (NaT)
invalid_pickup_dates = df['pickup_datetime'].isnull().sum()
invalid_dropoff_dates = df['dropoff_datetime'].isnull().sum()
print(f"Invalid (unparseable) pickup datetimes: {invalid_pickup_dates}")
print(f"Invalid (unparseable) dropoff datetimes: {invalid_dropoff_dates}")

# Check for illogical timestamps (dropoff before pickup)
invalid_duration_count = df[df['dropoff_datetime'] < df['pickup_datetime']].shape[0]
print(f"Trips with dropoff before pickup: {invalid_duration_count}")

# Check for missing GPS pings (already checked with df.isnull().sum(), reiterating if no missing values)
gps_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
missing_gps = df[gps_cols].isnull().sum().sum()
print(f"Missing GPS pings (sum across all relevant columns): {missing_gps}")

# Remove rows with invalid timestamps for further processing
df.dropna(subset=['pickup_datetime', 'dropoff_datetime'], inplace=True)
df = df[df['dropoff_datetime'] >= df['pickup_datetime']]
print(f"DataFrame shape after cleaning invalid timestamps: {df.shape}")

# --- 2. Feature Engineering ---
print("\n--- Feature Engineering ---")

# Hour-of-day
df['pickup_hour'] = df['pickup_datetime'].dt.hour
print("Created 'pickup_hour' feature.")

# Weekday/weekend
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek # Monday=0, Sunday=6
df['is_weekend'] = (df['pickup_dayofweek'] >= 5).astype(int)
print("Created 'pickup_dayofweek' and 'is_weekend' features.")

# Distance (Haversine formula)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c
    return distance

df['distance_km'] = df.apply(lambda row: haversine_distance(
    row['pickup_latitude'], row['pickup_longitude'],
    row['dropoff_latitude'], row['dropoff_longitude']
), axis=1)
print("Created 'distance_km' feature using Haversine formula.")

# Weather (convert to categorical for efficiency)
df['weather'] = df['weather'].astype('category')
print("Converted 'weather' column to categorical type.")

print("\nFeatures engineered successfully. New columns added.")
print(df.head())

# --- 3. Version the dataset ---
print("\n--- Dataset Versioning ---")
output_filename = 'NY_Trip_Processed_Data_v1.csv'
df.to_csv(output_filename, index=False)
print(f"Processed dataset saved to '{output_filename}'.")
print("Versioning complete.")


--- Schema Validation ---
Invalid (unparseable) pickup datetimes: 0
Invalid (unparseable) dropoff datetimes: 0
Trips with dropoff before pickup: 0
Missing GPS pings (sum across all relevant columns): 0
DataFrame shape after cleaning invalid timestamps: (1048575, 12)

--- Feature Engineering ---
Created 'pickup_hour' feature.
Created 'pickup_dayofweek' and 'is_weekend' features.
Created 'distance_km' feature using Haversine formula.
Converted 'weather' column to categorical type.

Features engineered successfully. New columns added.
          id  vendor_id     pickup_datetime    dropoff_datetime  \
0  id2875421          2 2016-03-14 17:24:00 2016-03-14 17:32:00   
1  id2377394          1 2016-06-12 00:43:00 2016-06-12 00:54:00   
2  id3858529          2 2016-01-19 11:35:00 2016-01-19 12:10:00   
3  id3504673          2 2016-04-06 19:32:00 2016-04-06 19:39:00   
4  id2181028          2 2016-03-26 13:30:00 2016-03-26 13:38:00   

   passenger_count   weather  pickup_longitude  pickup_lat

## Model Training and Comparison

In [23]:
import mlflow
from sklearn.ensemble import GradientBoostingRegressor

print("MLflow and GradientBoostingRegressor imported.")

ERROR: Operation cancelled by user
MLflow and GradientBoostingRegressor imported.


### Data Preparation for Modeling

In [30]:
# Define features (X) and target (y)
# Drop 'id', datetime columns, and original trip_duration as it's our target.
# One-hot encode 'weather' and 'store_and_fwd_flag'

X = df.drop(columns=['id', 'pickup_datetime', 'dropoff_datetime', 'trip_duration'])
y = df['trip_duration']

# Handle categorical features using one-hot encoding
X = pd.get_dummies(X, columns=['weather', 'store_and_fwd_flag', 'vendor_id'], drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print("Data split into training and testing sets.")

Training features shape: (838860, 16)
Testing features shape: (209715, 16)
Data split into training and testing sets.


### Model 1: Linear Regression

In [31]:
with mlflow.start_run(run_name="Linear_Regression_Model"):
    # Log parameters
    mlflow.log_param("model_type", "Linear Regression")

    # Initialize and train the model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    print("Linear Regression model trained.")

    # Make predictions
    y_pred_lr = lr_model.predict(X_test)

    # Evaluate the model
    mse_lr = mean_squared_error(y_test, y_pred_lr)
    r2_lr = r2_score(y_test, y_pred_lr)
    print(f"Linear Regression - Mean Squared Error: {mse_lr:.2f}")
    print(f"Linear Regression - R-squared: {r2_lr:.2f}")

    # Log metrics
    mlflow.log_metric("mse", mse_lr)
    mlflow.log_metric("r2", r2_lr)

    # Log the model
    mlflow.sklearn.log_model(lr_model, "linear_regression_model")
    print("Linear Regression model logged to MLflow.")

2026/08/09 16:31:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/09 16:31:28 INFO mlflow.store.db.utils: Updating database tables
2026/08/09 16:31:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Linear Regression model trained.
Linear Regression - Mean Squared Error: 9557237.30
Linear Regression - R-squared: 0.03
Linear Regression model logged to MLflow.


### Model 2: Gradient Boosting Regressor

In [33]:
with mlflow.start_run(run_name="Gradient_Boosting_Model"):
    # Log parameters
    mlflow.log_param("model_type", "Gradient Boosting Regressor")
    mlflow.log_param("n_estimators", 100) # Example hyperparameter
    mlflow.log_param("learning_rate", 0.1) # Example hyperparameter

    # Initialize and train the model
    gbr_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    gbr_model.fit(X_train, y_train)
    print("Gradient Boosting Regressor model trained.")

    # Make predictions
    y_pred_gbr = gbr_model.predict(X_test)

    # Evaluate the model
    mse_gbr = mean_squared_error(y_test, y_pred_gbr)
    r2_gbr = r2_score(y_test, y_pred_gbr)
    print(f"Gradient Boosting - Mean Squared Error: {mse_gbr:.2f}")
    print(f"Gradient Boosting - R-squared: {r2_gbr:.2f}")

    # Log metrics
    mlflow.log_metric("mse", mse_gbr)
    mlflow.log_metric("r2", r2_gbr)

    # Log the model
    mlflow.sklearn.log_model(gbr_model, "gradient_boosting_model")
    print("Gradient Boosting model logged to MLflow.")

Gradient Boosting Regressor model trained.


2026/08/09 16:40:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Gradient Boosting - Mean Squared Error: 11025336.11
Gradient Boosting - R-squared: -0.12
Gradient Boosting model logged to MLflow.


## Model Comparison Summary

By comparing the Mean Squared Error (MSE) and R-squared (R2) values logged by MLflow for both models, you can determine which model performed better for this dataset. MLflow provides a centralized platform to view and compare these metrics, as well as the hyperparameters used for each run.

In [35]:
print("\n--- Model Performance Comparison ---")
print(f"Linear Regression Model:")
print(f"  Mean Squared Error (MSE): {mse_lr:.2f}")
print(f"  R-squared (R2): {r2_lr:.2f}")
print("\n")
print(f"Gradient Boosting Regressor Model:")
print(f"  Mean Squared Error (MSE): {mse_gbr:.2f}")
print(f"  R-squared (R2): {r2_gbr:.2f}")

if r2_lr > r2_gbr:
    print("\nConclusion: Linear Regression appears to have performed slightly better based on R-squared.")
elif r2_gbr > r2_lr:
    print("\nConclusion: Gradient Boosting Regressor appears to have performed better based on R-squared.")
else:
    print("\nConclusion: Both models performed similarly based on R-squared.")


--- Model Performance Comparison ---
Linear Regression Model:
  Mean Squared Error (MSE): 9557237.30
  R-squared (R2): 0.03


Gradient Boosting Regressor Model:
  Mean Squared Error (MSE): 11025336.11
  R-squared (R2): -0.12

Conclusion: Linear Regression appears to have performed slightly better based on R-squared.
